In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/02 21:07:45 WARN Utils: Your hostname, cindi resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/02 21:07:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/02 21:07:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/02 21:08:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT**

In [3]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

[Stage 11:>                                                         (0 + 1) / 1]

Transaksi: 4000 | Cabang: 3 | Menu: 15


**B. TRANSFORM — Join & Kolom Turunan**

In [4]:
df_gabungan_tugas7 = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

df_gabungan_tugas7 = df_gabungan_tugas7.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan_tugas7.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. TRANSFORM — Window Function**

In [5]:
# Meringkas total penjualan per kombinasi cabang + menu terlebih dahulu
ringkasan_menu_cabang = df_gabungan_tugas7.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Window function untuk mengambil top-2 menu pada setiap cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

[Stage 24:>                                                         (0 + 1) / 1]

+-------------------------+---------+---------------+---------+
|nama_cabang              |nama_menu|total_penjualan|peringkat|
+-------------------------+---------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |
+-------------------------+---------+---------------+---------+



**D. TRANSFORM — Spark SQL**

In [9]:
df_gabungan_tugas7.createOrReplaceTempView("gabungan_tugas7")

penjualan_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan_tugas7
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. LOAD**

In [7]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

df_gabungan_tugas7.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.")
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

# Verifikasi: baca ulang lalu bandingkan jumlah baris
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris data gabungan (sebelum disimpan):", df_gabungan_tugas7.count())
print("Jumlah baris Parquet (setelah dibaca ulang) :", df_verifikasi.count())

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.
Found 4 items
-rw-r--r--   3 kiaa supergroup          0 2026-10-02 21:20 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - kiaa supergroup          0 2026-10-02 21:19 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - kiaa supergroup          0 2026-10-02 21:20 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - kiaa supergroup          0 2026-10-02 21:20 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


Jumlah baris data gabungan (sebelum disimpan): 4000


[Stage 39:>                                                         (0 + 3) / 3]

Jumlah baris Parquet (setelah dibaca ulang) : 4000


**F. Ringkasan Eksekutif & Rekomendasi**

In [8]:
ringkasan_eksekutif = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show(truncate=False)

[Stage 42:===================>                                      (1 + 2) / 3]

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



**ALASAN:** Berdasarkan analisis agregasi data, Kopi Nusantara Semarang tampil sebagai cabang berkinerja terbaik dengan total pendapatan tertinggi mencapai Rp90.542.000 dari 1.393 transaksi. Sebaliknya, Kopi Nusantara Yogyakarta mencatatkan kinerja terlemah dengan total penjualan terendah sebesar Rp84.645.000. Untuk mendongkrak performa cabang Yogyakarta, strategi pemasaran harus difokuskan pada promosi yang lebih gencar untuk Menu-12 dan Menu-15. Alasan utamanya adalah karena kedua menu tersebut terbukti menjadi top-2 menu paling diminati dengan kontribusi omzet terbesar di wilayah Yogyakarta. Mempromosikan menu unggulan ini melalui paket bundling atau diskon khusus akan secara efektif meningkatkan volume transaksi dan nilai belanja pelanggan.

**EKSPLORASI**

In [10]:
import time

# 1. Paksa tanpa Broadcast (Menggunakan SortMergeJoin / Shuffle)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

start = time.time()
df_tanpa_broadcast = df_transaksi.join(df_menu, "menu_id")
df_tanpa_broadcast.count()
print("Waktu SortMergeJoin:", time.time() - start)
df_tanpa_broadcast.explain()

# 2. Kembalikan & Paksa Hint Broadcast
from pyspark.sql.functions import broadcast

start = time.time()
df_dengan_broadcast = df_transaksi.join(broadcast(df_menu), "menu_id")
df_dengan_broadcast.count()
print("Waktu BroadcastJoin:", time.time() - start)
df_dengan_broadcast.explain()

Waktu SortMergeJoin: 4.04427433013916
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [menu_id#19, trx_id#17, cabang_id#18, qty#20, nama_menu#59, kategori_menu#60, harga#61]
   +- SortMergeJoin [menu_id#19], [menu_id#58], Inner
      :- Sort [menu_id#19 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(menu_id#19, 200), ENSURE_REQUIREMENTS, [plan_id=1976]
      :     +- Filter isnotnull(menu_id#19)
      :        +- FileScan csv [trx_id#17,cabang_id#18,menu_id#19,qty#20] Batched: false, DataFilters: [isnotnull(menu_id#19)], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv], PartitionFilters: [], PushedFilters: [IsNotNull(menu_id)], ReadSchema: struct<trx_id:string,cabang_id:int,menu_id:int,qty:int>
      +- Sort [menu_id#58 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(menu_id#58, 200), ENSURE_REQUIREMENTS, [plan_id=1977]
            +- Filter isnotnull(menu_id#58)

In [11]:
# Menguji filter di akhir query
df_test = df_transaksi.join(df_cabang, "cabang_id") \
                      .join(df_menu, "menu_id") \
                      .filter(col("qty") > 3)

df_test.explain()
# Amati bagian 'PushedFilters: [IsNotNull(qty), GreaterThanOrEqual(qty,4)]' pada Physical Plan!

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [menu_id#19, cabang_id#18, trx_id#17, qty#20, kepala_barista#34, kota#35, nama_cabang#36, nama_menu#59, kategori_menu#60, harga#61]
   +- SortMergeJoin [menu_id#19], [menu_id#58], Inner
      :- Sort [menu_id#19 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(menu_id#19, 200), ENSURE_REQUIREMENTS, [plan_id=2188]
      :     +- Project [cabang_id#18, trx_id#17, menu_id#19, qty#20, kepala_barista#34, kota#35, nama_cabang#36]
      :        +- SortMergeJoin [cast(cabang_id#18 as bigint)], [cabang_id#33L], Inner
      :           :- Sort [cast(cabang_id#18 as bigint) ASC NULLS FIRST], false, 0
      :           :  +- Exchange hashpartitioning(cast(cabang_id#18 as bigint), 200), ENSURE_REQUIREMENTS, [plan_id=2180]
      :           :     +- Filter (((isnotnull(qty#20) AND (qty#20 > 3)) AND isnotnull(cabang_id#18)) AND isnotnull(menu_id#19))
      :           :        +- FileScan csv [trx_id#17,cabang_id#18,m

In [12]:
from pyspark import StorageLevel

# Menggunakan persist dengan StorageLevel kustom
df_gabungan_tugas7.persist(StorageLevel.MEMORY_ONLY)

# Cek status penyimpanan
print("Is Cached:", df_gabungan_tugas7.is_cached)

# Jalankan action untuk mengisi cache
df_gabungan_tugas7.count()

# Bersihkan dari memori
df_gabungan_tugas7.unpersist()

Is Cached: True


DataFrame[menu_id: int, cabang_id: int, trx_id: string, qty: int, kepala_barista: string, kota: string, nama_cabang: string, nama_menu: string, kategori_menu: string, harga: int, total_penjualan: int]

In [13]:
df_parquet = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

# Query 1: Filter pada kolom partisi
print("=== QUERY DENGAN PARTITION PRUNING ===")
df_parquet.filter(col("kategori_menu") == "Kopi").explain()

# Query 2: Filter pada kolom non-partisi
print("=== QUERY TANPA PARTITION PRUNING ===")
df_parquet.filter(col("harga") > 20000).explain()

=== QUERY DENGAN PARTITION PRUNING ===
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [menu_id#850,cabang_id#851,trx_id#852,qty#853,kepala_barista#854,kota#855,nama_cabang#856,nama_menu#857,harga#858,total_penjualan#859,kategori_menu#860] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan], PartitionFilters: [isnotnull(kategori_menu#860), (kategori_menu#860 = Kopi)], PushedFilters: [], ReadSchema: struct<menu_id:int,cabang_id:int,trx_id:string,qty:int,kepala_barista:string,kota:string,nama_cab...


=== QUERY TANPA PARTITION PRUNING ===
== Physical Plan ==
*(1) Filter (isnotnull(harga#858) AND (harga#858 > 20000))
+- *(1) ColumnarToRow
   +- FileScan parquet [menu_id#850,cabang_id#851,trx_id#852,qty#853,kepala_barista#854,kota#855,nama_cabang#856,nama_menu#857,harga#858,total_penjualan#859,kategori_menu#860] Batched: true, DataFilters: [isnotnull(harga#858), (harga#858 > 20000)],